# Flood Prediction - Exploratory Data Analysis

This notebook documents the exploratory data analysis (EDA) performed on the historical
flood dataset before any model training took place. It mirrors Epic 1 and Epic 2 of the
project's Agile backlog: **Data Collection** and **Data Analysis**.

Run this notebook with the project's virtual environment active, from the project root,
so the relative path to `dataset/flood_dataset.xlsx` resolves correctly.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (9, 5)

## 2. Load Dataset

In [ ]:
df = pd.read_excel('dataset/flood_dataset.xlsx')
df.columns = [c.strip() for c in df.columns]
df.head()

In [ ]:
df.tail()

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', list(df.columns))

In [ ]:
df.describe()

In [ ]:
df.info()

## 3. Data Quality Checks

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
df['flood'].value_counts().rename('count').to_frame()

The target class is imbalanced (roughly 86% no-flood vs 14% flood years), which is expected
for a historical flood record - flood years are the exception, not the rule. This is why
precision/recall/F1 are tracked alongside accuracy during model comparison rather than relying
on accuracy alone.

## 4. Univariate Analysis

In [ ]:
numeric_cols = ['Temp', 'Humidity', 'Cloud Cover', 'ANNUAL', 'Jun-Sep']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='#1d4ed8')
    ax.set_title(f'Distribution of {col}')
axes.flatten()[-1].axis('off')
plt.tight_layout()
plt.savefig('reports/eda_histograms.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots()
sns.countplot(x='flood', data=df, palette=['#1d4ed8', '#dc2626'], ax=ax)
ax.set_xticklabels(['No Flood (0)', 'Flood (1)'])
ax.set_title('Class Distribution - Flood Occurrence')
plt.savefig('reports/eda_class_distribution.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['ANNUAL', 'Jun-Sep', 'Cloud Cover']):
    sns.boxplot(y=df[col], ax=ax, color='#38bdf8')
    ax.set_title(f'Boxplot - {col}')
plt.tight_layout()
plt.savefig('reports/eda_boxplots.png', dpi=150)
plt.show()

## 5. Multivariate Analysis

In [ ]:
plt.figure(figsize=(9, 7))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('reports/eda_correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
pairplot_cols = ['Temp', 'Humidity', 'Cloud Cover', 'ANNUAL', 'flood']
sns.pairplot(df[pairplot_cols], hue='flood', palette=['#1d4ed8', '#dc2626'], corner=True)
plt.savefig('reports/eda_pairplot.png', dpi=150)
plt.show()

## 6. Feature Importance (Preview)

A quick Random Forest fit on the raw (unscaled) features gives an early read on which signals
the model leans on most, ahead of the full training pipeline in `train_model.py`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

feature_cols = ['ANNUAL', 'Cloud Cover', 'Temp', 'Humidity', 'Jun-Sep']
X_preview = df[feature_cols]
y_preview = df['flood']

preview_model = RandomForestClassifier(n_estimators=200, random_state=42)
preview_model.fit(X_preview, y_preview)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': preview_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots()
sns.barplot(data=importance_df, x='importance', y='feature', color='#1d4ed8', ax=ax)
ax.set_title('Feature Importance (Preview Random Forest)')
plt.tight_layout()
plt.savefig('reports/eda_feature_importance.png', dpi=150)
plt.show()

importance_df

## 7. Next Steps

With the data understood, the actual training pipeline (missing value handling, IQR
outlier capping, scaling, train/test split, and training all four classifiers) lives in
`preprocessing/data_prep.py`, `training/*.py` and is orchestrated end-to-end by
`train_model.py` at the project root. Run:

```bash
python train_model.py
```

to reproduce the full model comparison and regenerate `models/floods.save`.